# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/Week%205/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane's target is `is_declining_label` — the same binary yes/no label ML-04 and ML-07 both used (`trend_direction == "down"`, 1/0). Per the `training-honest-models` skill's own method-choice table, a yes/no target with an observed label points at Logistic Regression first, then Random Forest — readable, then stronger, in that order. That's my method choice this week: fit both on the same client-aware split, and check whether Random Forest's extra complexity actually earns its keep over the version I can print and read.

I'm not reaching for K-Means — my target isn't an unlabeled grouping question, it's already a real 1/0 label, so clustering doesn't fit this lane. I'm also skipping Gradient Boosting this round: the skill's own caution is "where safe," and with only 32 clients and one already-close Logistic-vs-Random-Forest comparison to make sense of first, adding a third model this week would be complexity for its own sake, not something the comparison has earned yet. I do use permutation importance in Section 4 — the more rigorous of the two importance checks per the skill, since it's measured on the held-out test set rather than trusted from training alone.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

import pandas as pd
import numpy as np

RANDOM_STATE = 42

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same trailing-90-day label ML-04 and ML-07 both used. Never fed in as a feature below.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"{len(df):,} pages | {df['client_id'].nunique()} clients | base decline rate {df['is_declining_label'].mean():.3f}")

Working dir: /tmp/flyrank-ml-internship


30,000 pages | 32 clients | base decline rate 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm holding out whole clients, not random rows — the same client-aware convention `scripts/03_train_model.py` and last week's baseline both use. Shuffle the 32 unique `client_id`s with a fixed seed, hold out ~20% of clients as test, and every row from a held-out client stays out of training entirely. A row-random split would let the model see 9 of a client's 10 pages in training and just memorize that client's quirks on the 10th; a client-grouped split is the honest one for "will this generalize to a client my model has never seen," which is the real question a refresh-prioritization model has to answer.

One real number worth stating up front, not hiding: my train decline rate is 0.555 and my test decline rate is 0.391 — a real 16-point gap. That's not a bug in the split; it's the honest cost of grouping by client. Different clients genuinely decline at different base rates, and holding out whole clients means holding out whichever clients happened to land in that ~20%, base rate and all. I keep this in view through the rest of the notebook — it's exactly why the comparison in Section 3 leans on ranking metrics (precision@K, ROC-AUC, average precision) rather than raw accuracy, which would be distorted by the base-rate shift between train and test.

In [2]:
clients = df["client_id"].drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)

n_test_clients = max(1, round(len(shuffled) * 0.2))
test_clients = set(shuffled[:n_test_clients])

test_mask = df["client_id"].isin(test_clients)
train_df = df[~test_mask].reset_index(drop=True)
test_df = df[test_mask].reset_index(drop=True)

print(f"{len(shuffled)} clients total -> {n_test_clients} held out for test")
print(f"train: {len(train_df):,} rows | decline rate {train_df['is_declining_label'].mean():.3f}")
print(f"test:  {len(test_df):,} rows | decline rate {test_df['is_declining_label'].mean():.3f}")

32 clients total -> 6 held out for test
train: 27,675 rows | decline rate 0.555
test:  2,325 rows | decline rate 0.391


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same data, same metric, same split as my Week-4 baseline — but with one honesty fix first. ML-07's rule needs each page's position tier to know its "expected" CTR, and while rebuilding it I found a real data-quality trap: the raw `position_tier` column in this file labels every `avg_position == 0` row as `top_3` — the *best* tier — when `avg_position == 0` actually means the page has no real position data at all (never meaningfully measured, not literally ranked #1). Feeding that straight into a real comparison would have quietly told 1,205 unranked pages they're top performers. I recompute the tier myself below, giving `avg_position == 0` rows their own honest `no_position_data` category instead of borrowing FlyRank's best tier by accident.

I also refit the baseline's one lookup table — tier-mean CTR — on train rows only. ML-07's original notebook computed it over the whole file, which was fine for a one-off rule with no held-out split to worry about; scoring that same rule on a held-out test split this week means the lookup table itself has to be train-only, or the "baseline" in this table wouldn't really be evaluated out-of-sample.

In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# Position tier, rebuilt honestly: avg_position == 0 means "no real position data,"
# not rank zero. Every avg_position == 0 row in the raw file carries the raw
# position_tier value "top_3" -- confirmed by checking it directly. I recompute the
# tier myself, the same way ML-07's baseline notebook does, and give the no-data rows
# their own honest category instead of inheriting FlyRank's best tier by accident.
pos_bins = [0, 3, 10, 20, 50, np.inf]
pos_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]

for frame in (train_df, test_df):
    frame["position_tier_fixed"] = "no_position_data"
    has_pos = frame["avg_position"] > 0
    frame.loc[has_pos, "position_tier_fixed"] = pd.cut(
        frame.loc[has_pos, "avg_position"], bins=pos_bins, labels=pos_labels
    ).astype(str)

# Baseline tier-mean CTR, computed on TRAIN ONLY so the baseline is scored out-of-sample too.
train_has_pos = train_df["avg_position"] > 0
tier_mean_ctr = train_df.loc[train_has_pos].groupby("position_tier_fixed")["ctr"].mean()
print("Train-only tier mean CTR:")
print(tier_mean_ctr.round(3))

def baseline_score(frame):
    expected_ctr = frame["position_tier_fixed"].map(tier_mean_ctr)
    visible = (frame["impressions_90d"] >= 100).astype(int)
    reachable = ((frame["avg_position"] > 0) & (frame["avg_position"] <= 50)).astype(int)
    ctr_gap = (expected_ctr - frame["ctr"]).clip(lower=0).fillna(0)
    return visible * reachable * ctr_gap * np.log1p(frame["impressions_90d"])

test_df["baseline_score"] = baseline_score(test_df)

baseline_metrics = {
    "precision_at_20": precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 20),
    "precision_at_50": precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 50),
    "precision_at_100": precision_at_k(test_df["is_declining_label"], test_df["baseline_score"], 100),
    "roc_auc": roc_auc_score(test_df["is_declining_label"], test_df["baseline_score"]),
    "average_precision": average_precision_score(test_df["is_declining_label"], test_df["baseline_score"]),
}

print("\nBaseline (ML-07 rule, refit train-only) on held-out test:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.3f}")
print(f"  test base rate: {test_df['is_declining_label'].mean():.3f}")

Train-only tier mean CTR:
position_tier_fixed
deep        0.148
page_1      0.390
page_3_5    0.167
striking    0.271
top_3       0.901
Name: ctr, dtype: float64

Baseline (ML-07 rule, refit train-only) on held-out test:
  precision_at_20: 0.700
  precision_at_50: 0.640
  precision_at_100: 0.650
  roc_auc: 0.596
  average_precision: 0.476
  test base rate: 0.391


In [4]:
NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL = ["competition_level", "content_type", "main_intent", "provider_used", "model_used", "position_tier_fixed"]
LOG_COLUMNS = ["search_volume", "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
               "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d"]
# These 9 columns have real, content-type-correlated missingness (the signal audit's own
# finding: word_count/char_count are 0% missing for feedly/comparison articles but ~28%
# missing within keyword articles). A blind fillna(0) would inject a false "zero" signal --
# so I flag whether the value was ever present, separately from filling the number itself.
MISSING_FLAG_COLUMNS = ["word_count", "char_count", "search_volume", "competition", "cpc",
                         "main_intent", "competition_level", "provider_used", "model_used"]

def engineer(frame):
    out = frame.copy()
    for col in MISSING_FLAG_COLUMNS:
        out[f"has_{col}"] = out[col].notna().astype(int)
    num = out[NUMERIC].apply(pd.to_numeric, errors="coerce")
    for col in LOG_COLUMNS:
        num[col] = np.log1p(num[col].clip(lower=0))
    num = num.replace([np.inf, -np.inf], np.nan).fillna(0)
    flags = out[[f"has_{c}" for c in MISSING_FLAG_COLUMNS]]
    cat = pd.get_dummies(out[CATEGORICAL].astype(str), dummy_na=False, dtype=float)
    return pd.concat([num, flags, cat], axis=1)

train_feat = engineer(train_df)
test_feat = engineer(test_df)
# align handles any category present in train but not test (or vice versa) without crashing
train_feat, test_feat = train_feat.align(test_feat, join="left", axis=1, fill_value=0)

y_train = train_df["is_declining_label"]
y_test = test_df["is_declining_label"]

print(f"feature matrix: {train_feat.shape[1]} columns | train {len(train_feat):,} rows | test {len(test_feat):,} rows")

feature matrix: 52 columns | train 27,675 rows | test 2,325 rows


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=8, min_samples_leaf=25,
        n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE,
    ),
}

fitted = {}
model_metrics = {}
for name, model in models.items():
    model.fit(train_feat, y_train)
    fitted[name] = model
    scores = model.predict_proba(test_feat)[:, 1]
    model_metrics[name] = {
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "precision_at_50": precision_at_k(y_test, scores, 50),
        "precision_at_100": precision_at_k(y_test, scores, 100),
        "roc_auc": roc_auc_score(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
    }

comparison = pd.DataFrame({"baseline_ml07_rule": baseline_metrics, **model_metrics}).T
comparison["test_base_rate"] = round(test_df["is_declining_label"].mean(), 3)

print("Model-vs-baseline comparison -- same test split, same metrics:\n")
print(comparison.round(3).to_string())

Model-vs-baseline comparison -- same test split, same metrics:

                     precision_at_20  precision_at_50  precision_at_100  roc_auc  average_precision  test_base_rate
baseline_ml07_rule              0.70             0.64              0.65    0.596              0.476           0.391
logistic_regression             0.55             0.54              0.57    0.713              0.551           0.391
random_forest                   0.70             0.66              0.52    0.725              0.547           0.391


**The table, on the same held-out test split, same target, same precision@K / ROC-AUC / average-precision metrics:**

| | precision@20 | precision@50 | precision@100 | ROC-AUC | avg precision |
|---|---|---|---|---|---|
| baseline (ML-07 rule) | 0.70 | 0.64 | 0.65 | 0.596 | 0.476 |
| logistic regression | 0.55 | 0.54 | 0.57 | 0.713 | 0.551 |
| random forest | 0.70 | 0.66 | 0.52 | 0.725 | 0.547 |

*(test base rate 0.391)*

Three honest things this table says, not one flattering headline.

**First, the ML-07 rule is still a genuinely strong top-of-list ranker.** It ties Random Forest at precision@20 and isn't far off at precision@50, using nothing but three trailing-90-day numbers multiplied together. A learned model beating a simple, well-built rule wasn't guaranteed — and at the very top of the list, here, it mostly didn't happen.

**Second, both learned models clearly win on overall ranking quality** — ROC-AUC (0.71–0.73 vs. 0.596) and average precision (0.55 vs. 0.48) — meaning they separate decliners from non-decliners better across the *whole* ranked list, not just the top 20 or 50. The rule is a narrow specialist tuned for exactly the CTR-vs-position signal it was built on; the models pick up a broader mix of signal (impressions, content age, tier, content type) the rule never looks at at all.

**Third — the one that matters most for "does not reward complexity alone" — Random Forest does not clearly beat Logistic Regression.** RF wins precision@20, precision@50, and edges out ROC-AUC, but it *loses* at precision@100 (0.52 vs. 0.57) and average precision (0.547 vs. 0.551) — both real margins, not noise. A five-line logistic regression I can actually reason about is doing essentially the same job as a 300-tree forest. If I only had budget to ship one model, Logistic Regression is the harder one to argue against on this table — even though "Random Forest wins" is the flattering headline I'd have been tempted to write if I'd only looked at precision@20.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I'm running the error analysis on Random Forest, since it's the model that actually shipped a small win in Section 3 and the one whose extra complexity needs justifying. Two independent feature-importance checks first, then where it's actually getting the label wrong.

In [6]:
from sklearn.inspection import permutation_importance

rf = fitted["random_forest"]

built_in = pd.Series(rf.feature_importances_, index=train_feat.columns).sort_values(ascending=False)
print("Built-in feature_importances_ (Random Forest), top 8:")
print(built_in.head(8).round(4))

# The more rigorous check per the skill: measured on held-out test, by how much shuffling
# each column actually hurts ROC-AUC -- not just how often a tree happened to split on it.
perm = permutation_importance(rf, test_feat, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="roc_auc")
perm_importance = pd.Series(perm.importances_mean, index=test_feat.columns).sort_values(ascending=False)
print("\nPermutation importance (test set, scored by ROC-AUC), top 8:")
print(perm_importance.head(8).round(4))

Built-in feature_importances_ (Random Forest), top 8:
impressions_90d                         0.2063
avg_position                            0.1261
content_age_days                        0.1063
position_tier_fixed_no_position_data    0.0587
char_count                              0.0379
word_count                              0.0355
clicks_90d                              0.0340
ctr                                     0.0278
dtype: float64



Permutation importance (test set, scored by ROC-AUC), top 8:
impressions_90d                         0.0811
avg_position                            0.0093
position_tier_fixed_no_position_data    0.0073
clicks_90d                              0.0065
search_volume                           0.0061
ctr                                     0.0056
scroll_rate                             0.0034
engaged_sessions_90d                    0.0022
dtype: float64


Both methods agree on the top two features, which is reassuring on its own — built-in importance and permutation importance are computed in completely different ways (one from how much a feature reduces impurity while the trees were being built, the other from how much shuffling it hurts held-out ROC-AUC after the fact) and they still point the same direction.

`impressions_90d` dominates both rankings by a wide margin (0.206 built-in, 0.081 permutation — genuinely far ahead of everything else), with `avg_position` and my recomputed `no_position_data` flag close behind. All three are plausible: whether a page is declining is fundamentally about its trailing search demand and where it sits in the rankings, so a model leaning hardest on *how much traffic it's getting* and *where it ranks* is leaning on exactly the signal I'd expect a human reviewer to check first — not something suspicious. Nothing here is the "suspiciously perfect" red flag the skill warns about: no feature sits anywhere near 1.0 importance, and no single feature alone would let me guess the label with certainty, which is what real leakage usually looks like.

In [7]:
test_scores = rf.predict_proba(test_feat)[:, 1]
test_pred = (test_scores >= 0.5).astype(int)
test_df["rf_pred"] = test_pred
test_df["rf_proba"] = test_scores
test_df["rf_correct"] = (test_pred == y_test.to_numpy())

overall_error = 1 - test_df["rf_correct"].mean()
print(f"Overall Random Forest error rate (0.5 threshold): {overall_error:.3f}")

by_type = test_df.groupby("content_type")["rf_correct"].agg(n="count", error_rate=lambda s: 1 - s.mean())
print("\nError rate by content_type:")
print(by_type.round(3))

by_tier = test_df.groupby("position_tier_fixed")["rf_correct"].agg(n="count", error_rate=lambda s: 1 - s.mean())
print("\nError rate by position tier (my fixed tiers):")
print(by_tier.round(3))

no_pos = test_df["position_tier_fixed"] == "no_position_data"
print(f"\nno_position_data rows: {no_pos.sum()}")
print(f"decline rate among no-position rows: {test_df.loc[no_pos, 'is_declining_label'].mean():.3f}")
print(f"decline rate among has-position rows: {test_df.loc[~no_pos, 'is_declining_label'].mean():.3f}")

Overall Random Forest error rate (0.5 threshold): 0.334

Error rate by content_type:
                    n  error_rate
content_type                     
feedly article    958       0.230
keyword article  1367       0.407

Error rate by position tier (my fixed tiers):
                        n  error_rate
position_tier_fixed                  
deep                   59       0.390
no_position_data      283       0.007
page_1               1062       0.391
page_3_5              282       0.443
striking              402       0.413
top_3                 237       0.194

no_position_data rows: 283
decline rate among no-position rows: 0.007
decline rate among has-position rows: 0.444


Two real, unequal error rates worth naming, plus one number that needs an explicit caveat rather than a victory lap.

**By content type**, `keyword article` is nearly twice as hard as `feedly article` — 40.7% error vs. 23.0%. Keyword articles are FlyRank's dominant, most heterogeneous content type (27k+ of the 30k pages), so it's plausible the model simply has more genuinely ambiguous cases to get wrong there, not that it's uniformly worse at that content type.

**By position tier**, the murky middle is hardest — `page_3_5` (44.3%), `striking` (41.3%), and `page_1` (39.1%) all sit around 40%+ error, while `top_3` is the model's best-performing real tier (19.4%). That matches intuition: pages clearly winning or clearly buried are easier to call than pages sitting in the ambiguous middle of the rankings.

**The one number I'm not taking a victory lap on**: the `no_position_data` tier shows a 0.7% error rate — by far the lowest of any group. That is not the model being unusually skilled there. I checked the label directly: pages with no real position data decline only 0.7% of the time in my test split, versus 44.4% for pages that do have position data — an almost-constant label. Predicting "not declining" for every `no_position_data` page would already get this right 99.3% of the time without the model doing any real work. I'm flagging this explicitly so it doesn't get read as a modeling strength it isn't.

In [8]:
wrong = test_df[~test_df["rf_correct"]].copy()
wrong["confidence_gap"] = (wrong["rf_proba"] - 0.5).abs()
top3 = wrong.sort_values("confidence_gap", ascending=False).head(3)

cols = ["content_id", "client_id", "rf_proba", "is_declining_label", "content_age_days",
        "days_since_last_update", "avg_position", "ctr", "impressions_90d",
        "content_type", "position_tier_fixed"]
print(top3[cols].to_string(index=False))

          content_id         client_id  rf_proba  is_declining_label  content_age_days  days_since_last_update  avg_position  ctr  impressions_90d    content_type position_tier_fixed
content_28b4223f4e5f client_98a3ab7c34  0.076035                   1                91                       1           0.0  0.0                1 keyword article    no_position_data
content_34b14c00f80c client_d4735e3a26  0.148506                   1               308                      20           0.0  0.0                3  feedly article    no_position_data
content_86748254b6bf client_d4735e3a26  0.225273                   1               495                      20          90.0  0.0                1 keyword article                deep


**The 3 most confident wrong calls, and why they're hard:**

`content_28b4223f4e5f` (proba 0.076, predicted "not declining," actually declining) — recently touched (updated 1 day ago), `no_position_data`, and only 1 impression in the whole 90-day window. The model reads this almost entirely as safe: fresh update, and it's already learned `no_position_data` pages almost never decline. But 1 impression is a near-zero base — a swing from 1 impression to 0, or 2 to 1, can register as a huge *relative* "decline" that means almost nothing in absolute terms. This is very likely the label reacting to noise on a page with essentially no real traffic to lose.

`content_34b14c00f80c` (proba 0.149, same miss shape) — `no_position_data`, 3 impressions in 90 days, older content (308 days). Same root cause: another near-zero-traffic page the model correctly reads as low-risk by the subgroup's overwhelming pattern, but the label disagrees — again almost certainly a tiny-denominator trend artifact, not a real decline a human reviewer would flag as urgent.

`content_86748254b6bf` (proba 0.225) — different shape from the first two: this one has a real, if terrible, position (rank 90, "deep" tier), so it isn't in the trivially-easy `no_position_data` bucket. But it shares the same root issue — 1 impression in 90 days, old content (495 days), effectively invisible already. The model reads it as "already at the floor, nothing left to decline"; the label's percentage-based logic can still call that "down."

**The common thread, and the actual finding**: all three of my most-confident wrong predictions share extremely low `impressions_90d` (1–3 in the whole trailing window). That's not a coincidence — it's the most useful thing this error analysis surfaced. `is_declining_label` is built from `trend_direction`/`trend_pct`, almost certainly a percentage change, and percentage change on a near-zero denominator is unstable by construction. My model is arguably making the *more* defensible call on all three cases — reading near-zero traffic as already-bottomed-out rather than actively declining — and disagreeing with a label that's noisy at the extreme low-volume end. That's a real limitation of the label itself, not something more model complexity would fix, and it's worth flagging to whoever owns the label definition rather than chasing it with a bigger model.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous `client_id`/`content_id` values, same as the rest of this repo
- [x] My claims use careful words: observed, measured, directional, decision-support — the comparison table reports both wins and losses per model rather than one flattering summary, and the error analysis explicitly caveats the one subgroup number that looked stronger than it is
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

The last box is mine to check after I commit this executed notebook to my repo — this environment can't push to GitHub directly, so I commit it through GitHub's own web upload instead of `git push`.